# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

### Obtención del corpus
Primero descargamos nltk para obtener el corpus de Project Gutneberg

In [18]:
%pip install nltk pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import nltk
nltk.download('gutenberg')
nltk.download('stopwords')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
import re
import pandas as pd
from nltk.corpus import gutenberg,stopwords
stopwords_ingles = set(stopwords.words("english"))
def preprocesar(texto):
    #primero normalizamos el texto
    texto = texto.lower()
    #extraemos las secuencias de letras
    terminos = re.findall(r"[a-z]+",texto)
    #eliminamos las stopwords
    terminos = [termino for termino in terminos if termino not in stopwords_ingles]
    return terminos
# guardamos los documentos originales
documentos = {}
for identificador in gutenberg.fileids():
    texto_original = gutenberg.raw(identificador)
    terminos_procesados = preprocesar(texto_original)
    documentos[identificador] = {
        "texto_original": texto_original,
        "terminos_procesados": terminos_procesados
    }
print("Documentos preprocesados", len(documentos))


Documentos preprocesados 18


### Construimos la tabla solicitada

In [21]:
filas = []
for identificador, datos in documentos.items():
    texto = datos["texto_original"]
    encabezado = re.search(r"\[([^\]]+)\]",texto[:500])
    if encabezado:
        titulo = encabezado.group(1)
    else:
        titulo = "Título no encontrado"
    filas.append(
        {
            "identificador de documento": identificador,
            "titulo": titulo,
            "Número de términos únicos": len(set(datos["terminos_procesados"])),
        }
    )
tabla_documentos = pd.DataFrame(filas)
display(tabla_documentos)

,identificador de documento,titulo,Número de términos únicos
0,austen-emma.txt,Emma by Jane Austen 1816,6965
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,5607
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,6148
3,bible-kjv.txt,The King James Bible,12443
4,blake-poems.txt,Poems by William Blake 1789,1400
5,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryan...,3696
6,burgess-busterbrown.txt,The Adventures of Buster Bear by Thornton W. B...,1382
7,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carr...,2423
8,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,8029
9,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton...,7589


### Mostramos características del corpus

In [22]:
numero_documentos = len(documentos)
longitudes = []
vocabulario = set()
for identificador,datos in documentos.items():
    terminos = datos["terminos_procesados"]
    longitud = len(terminos)
    longitudes.append(longitud)
    vocabulario.update(terminos)
longitud_minima = min(longitudes)
longitud_media= sum(longitudes)/len(longitudes)
longitud_maxima = max(longitudes)
print("Número de documentos:", numero_documentos)
print("tamaño del vocabulario:", len(vocabulario))
print("longitud mínima:", longitud_minima)
print("longitud media:", longitud_media)
print("longitud maxima:", longitud_maxima)



Número de documentos: 18
tamaño del vocabulario: 41357
longitud mínima: 3805
longitud media: 56987.5
longitud maxima: 374945


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [23]:
from collections import Counter
import math
def calcular_tf(terminos):
    return Counter(terminos)
tf_documentos= {}
for identificador, datos in documentos.items():
    terminos = datos["terminos_procesados"]
    tf_documentos[identificador] = calcular_tf(terminos)
def calcular_df(documentos):
    df = Counter()
    for datos in documentos.values():
        terminos = datos["terminos_procesados"]
        terminos_unicos = set(terminos)
        df.update(terminos_unicos)
    return df
df_corpus = calcular_df(documentos)
def calcular_idf(df_corpus, numero_documentos):
    idf = {}
    for termino, frecuencia_documentos in df_corpus.items():
        idf[termino] = math.log(numero_documentos / frecuencia_documentos) +1
    return idf
numero_documentos = len(documentos)
idf_corpus = calcular_idf(df_corpus, numero_documentos)

def calcular_tfidf(tf_documentos, idf_corpus):
    tfidf_documentos = {}

    for identificador, frecuencias in tf_documentos.items():
        pesos = {}

        for termino, tf in frecuencias.items():
            pesos[termino] = tf * idf_corpus[termino]

        tfidf_documentos[identificador] = pesos

    return tfidf_documentos
tfidf_documentos = calcular_tfidf(tf_documentos, idf_corpus)

#imprimimos ejemplos
identificador = "austen-emma.txt"
termino = "emma"

print("TF:", tf_documentos[identificador][termino])
print("DF:", df_corpus[termino])
print("IDF:", idf_corpus.get(termino))
print("TF-IDF:", tfidf_documentos[identificador].get(termino, 0))


TF: 865
DF: 2
IDF: 3.1972245773362196
TF-IDF: 2765.59925939583


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

### Vocabulario ordenado

In [24]:
ids_documentos = list(documentos.keys())
# utilizamos sorted para ordenar
vocabulario_ordenado = sorted(vocabulario)
#imprimimos un ejemplo
print("Cantidad de documentos:", len(ids_documentos))
print("Cantidad de términos:", len(vocabulario_ordenado))
print("Primeros 10 términos:", vocabulario_ordenado[:10])


Cantidad de documentos: 18
Cantidad de términos: 41357
Primeros 10 términos: ['aaron', 'aaronites', 'ab', 'aback', 'abaddon', 'abaft', 'abagtha', 'abana', 'abandon', 'abandoned']


### Contar las frecuencias TF

In [25]:

tf_documentos = {}
for identificador in ids_documentos:
    terminos = documentos[identificador]["terminos_procesados"]
    tf_documentos[identificador] = calcular_tf(terminos)
# imprimimos un ejemplo
print(tf_documentos["austen-emma.txt"]["emma"])

865


### Calcular DF contando cada documento una sola vez por término.

In [26]:
df_corpus = calcular_df(documentos)
# imprimimos un ejemplo
print("DF de 'emma':", df_corpus["emma"])

DF de 'emma': 2


### Calcular IDF con la función de la sección anterior.

In [27]:
numero_documentos = len(ids_documentos)
idf_corpus = calcular_idf(df_corpus, numero_documentos)
#imprimimos un ejemplo
print("IDF de 'emma':", idf_corpus.get("emma", "No encontrado"))

IDF de 'emma': 3.1972245773362196


### Obtener la matriz documento-término TF-IDF.

In [28]:
from scipy.sparse import csr_matrix
filas = []
columnas = []
valores = []
indice_terminos = {}
for indice, termino in enumerate(vocabulario_ordenado):
    indice_terminos[termino] = indice

for indice_documento, identificador in enumerate(ids_documentos):
    # Counter con los TF del documento actual
    frecuencias = tf_documentos[identificador]
    for termino, tf in frecuencias.items():
        indice_termino = indice_terminos[termino]
        # Peso TF-IDF = TF × IDF
        peso = tf * idf_corpus[termino]
        filas.append(indice_documento)
        columnas.append(indice_termino)
        valores.append(peso)
matriz_tfidf = csr_matrix(
    (valores, (filas, columnas)),
    shape=(len(ids_documentos), len(vocabulario_ordenado))
)
print("Dimensiones de la matriz:", matriz_tfidf.shape)

Dimensiones de la matriz: (18, 41357)


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
corpus_procesado = []
for identificador in ids_documentos:
    terminos = documentos[identificador]["terminos_procesados"]
    corpus_procesado.append(terminos)
def devolver_terminos(terminos):
    return terminos

vectorizador = TfidfVectorizer(
    analyzer=devolver_terminos,
    vocabulary=indice_terminos,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)
matriz_sklearn = vectorizador.fit_transform(corpus_procesado)
print("Dimensiones de tu matriz:", matriz_tfidf.shape)
print("Dimensiones de scikit-learn:", matriz_sklearn.shape)
diferencia = matriz_tfidf - matriz_sklearn
if diferencia.nnz == 0:
    diferencia_maxima = 0.0
else:
    diferencia_maxima = abs(diferencia.data).max()

print("Máxima diferencia absoluta:", diferencia_maxima)

Dimensiones de tu matriz: (18, 41357)
Dimensiones de scikit-learn: (18, 41357)
Máxima diferencia absoluta: 0.0


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

### Definimos las consultas

In [30]:
consultas = {
    "Q1":{
        "texto": "love and marriage",
        "necesidad": "Encontrar textos sobre el amor y matrimonio"
    },
    "Q2":{
        "texto": "sea and whale",
        "necesidad": "Encontrar textos relacionados con el mar y las ballenas"
    },
    "Q3":{
        "texto":"king and kingdom",
        "necesidad": "Encontrar textos sobre reyes y reinos"
    },
    "Q4":{
        "texto": "children and family",
        "necesidad": "Encontrar textos relacionados con niños y familia"
    }
}
for identificador, consulta in consultas.items():
    print(f"{identificador}: {consulta['texto']}")
    print(f"Necesidad: {consulta['necesidad']}\n")


Q1: love and marriage
Necesidad: Encontrar textos sobre el amor y matrimonio

Q2: sea and whale
Necesidad: Encontrar textos relacionados con el mar y las ballenas

Q3: king and kingdom
Necesidad: Encontrar textos sobre reyes y reinos

Q4: children and family
Necesidad: Encontrar textos relacionados con niños y familia



### Procesar las consultas

In [33]:
ids_consultas = list(consultas.keys())
consultas_procesadas = []
for identificador in ids_consultas:
    texto = consultas[identificador]["texto"]
    terminos = preprocesar(texto)
    consultas_procesadas.append(terminos)
    print(f"{identificador}: {texto}")
    print("terminos procesados:", terminos)



Q1: love and marriage
terminos procesados: ['love', 'marriage']
Q2: sea and whale
terminos procesados: ['sea', 'whale']
Q3: king and kingdom
terminos procesados: ['king', 'kingdom']
Q4: children and family
terminos procesados: ['children', 'family']


### Construimos los vectores TF-IDF

In [34]:
matriz_consultas = vectorizador.transform(consultas_procesadas)

print("Dimensiones:", matriz_consultas.shape)

Dimensiones: (4, 41357)


### Realizamos la similutd coseno

In [35]:
import numpy as np

def similitud_coseno_manual(vector_consulta, vector_documento):
    producto_punto = vector_consulta.multiply(vector_documento).sum()
    norma_consulta = np.sqrt(
        vector_consulta.multiply(vector_consulta).sum()
    )

    norma_documento = np.sqrt(
        vector_documento.multiply(vector_documento).sum()
    )
    if norma_consulta == 0 or norma_documento == 0:
        return 0.0

    similitud = producto_punto / (norma_consulta * norma_documento)

    return float(similitud)

### Se calculan las similitudes

In [38]:
similitudes_manuales = np.zeros(
    (len(ids_consultas), len(ids_documentos))
)

for i in range(len(ids_consultas)):
    for j in range(len(ids_documentos)):
        vector_consulta = matriz_consultas.getrow(i)
        vector_documento = matriz_tfidf.getrow(j)

        similitudes_manuales[i, j] = similitud_coseno_manual(
            vector_consulta,
            vector_documento
        )

tabla_similitudes = pd.DataFrame(
    similitudes_manuales,
    index=ids_consultas,
    columns=ids_documentos
)

display(tabla_similitudes)

Dimensiones: (4, 18)


,austen-emma.txt,austen-persuasion.txt,austen-sense.txt,bible-kjv.txt,blake-poems.txt,bryant-stories.txt,burgess-busterbrown.txt,carroll-alice.txt,chesterton-ball.txt,chesterton-brown.txt,chesterton-thursday.txt,edgeworth-parents.txt,melville-moby_dick.txt,milton-paradise.txt,shakespeare-caesar.txt,shakespeare-hamlet.txt,shakespeare-macbeth.txt,whitman-leaves.txt
Q1,0.020386,0.022337,0.023595,0.007817,0.094929,0.008351,0.000727,0.001646,0.006006,0.004312,0.004259,0.015831,0.003827,0.042289,0.000773,0.004401,0.000000,0.096663
Q2,0.001184,0.004040,0.000000,0.006558,0.002155,0.030560,0.000000,0.005303,0.010623,0.020508,0.007753,0.001317,0.470473,0.015645,0.000756,0.004677,0.001545,0.048863
Q3,0.001184,0.000555,0.000338,0.065730,0.002651,0.077454,0.000000,0.029354,0.002519,0.008544,0.001589,0.002430,0.008322,0.020596,0.001859,0.075817,0.034837,0.001147
Q4,0.018924,0.038711,0.023989,0.037296,0.029640,0.023518,0.000575,0.005104,0.004712,0.033577,0.003649,0.026792,0.004412,0.002089,0.001299,0.001232,0.004722,0.020514


### Comparación

In [39]:
from sklearn.metrics.pairwise import cosine_similarity

similitudes_sklearn = cosine_similarity(
    matriz_consultas,
    matriz_tfidf
)

diferencia_maxima = np.max(
    np.abs(similitudes_manuales - similitudes_sklearn)
)

print("Máxima diferencia absoluta:", diferencia_maxima)

Máxima diferencia absoluta: 1.5543122344752192e-15


### Cálculo de una consulta

In [41]:
id_consulta = "Q2"
id_documento = "melville-moby_dick.txt"

fila_consulta = ids_consultas.index(id_consulta)
fila_documento = ids_documentos.index(id_documento)

vector_consulta = matriz_consultas.getrow(fila_consulta)
vector_documento = matriz_tfidf.getrow(fila_documento)

productos = vector_consulta.multiply(vector_documento).tocsr()

columnas_compartidas = productos.indices

filas_detalle = []

for columna in columnas_compartidas:
    termino = vocabulario_ordenado[columna]

    peso_consulta = float(vector_consulta[0, columna])
    peso_documento = float(vector_documento[0, columna])
    producto_parcial = peso_consulta * peso_documento

    filas_detalle.append({
        "Término compartido": termino,
        "Peso en consulta": peso_consulta,
        "Peso en documento": peso_documento,
        "Producto parcial": producto_parcial
    })

tabla_detalle = pd.DataFrame(
    filas_detalle,
    columns=[
        "Término compartido",
        "Peso en consulta",
        "Peso en documento",
        "Producto parcial"
    ]
)

display(tabla_detalle)

,Término compartido,Peso en consulta,Peso en documento,Producto parcial
0,sea,1.117783,508.591281,568.494706
1,whale,2.098612,2572.898666,5399.516758


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

### Verificamos las puntuaciones

In [44]:
import numpy as np

tolerancia = 1e-10

diferencia_maxima = np.max(
    np.abs(similitudes_manuales - similitudes_sklearn)
)

print("Máxima diferencia entre puntuaciones:", diferencia_maxima)

if diferencia_maxima <= tolerancia:
    print("Las puntuaciones coinciden dentro de la tolerancia.")
else:
    print("Las puntuaciones presentan diferencias. Revisa el cálculo.")
titulos_documentos = dict(
    zip(
        tabla_documentos["identificador de documento"],
        tabla_documentos["titulo"]
    )
)
print(tabla_documentos.columns)

Máxima diferencia entre puntuaciones: 1.5543122344752192e-15
Las puntuaciones coinciden dentro de la tolerancia.
Index(['identificador de documento', 'titulo', 'Número de términos únicos'], dtype='str')


### función para obtener un fragmento

In [46]:
def obtener_fragmento(texto_original, terminos_consulta, contexto=100):
    primera_posicion = None

    for termino in terminos_consulta:
        coincidencia = re.search(
            re.escape(termino),
            texto_original,
            flags=re.IGNORECASE
        )
        if coincidencia:
            posicion = coincidencia.start()

            if primera_posicion is None or posicion < primera_posicion:
                primera_posicion = posicion
    if primera_posicion is None:
        return " ".join(texto_original[:2 * contexto].split())
    inicio = max(0, primera_posicion - contexto)
    fin = min(len(texto_original), primera_posicion + contexto)
    fragmento = texto_original[inicio:fin]
    fragmento = " ".join(fragmento.split())

    return "..." + fragmento + "..."

### Construir el ranking

In [48]:
def construir_ranking(
    indice_consulta,
    similitudes,
    ids_documentos,
    documentos,
    titulos_documentos,
    consultas_procesadas,
    top_k=5
):
    puntuaciones = similitudes[indice_consulta]
    indices_ordenados = sorted(
        range(len(ids_documentos)),
        key=lambda i: (-puntuaciones[i], ids_documentos[i])
    )

    filas_ranking = []

    for posicion, indice_documento in enumerate(
        indices_ordenados[:top_k],
        start=1
    ):
        identificador = ids_documentos[indice_documento]
        puntuacion = puntuaciones[indice_documento]

        texto_original = documentos[identificador]["texto_original"]
        terminos_consulta = consultas_procesadas[indice_consulta]

        fragmento = obtener_fragmento(
            texto_original,
            terminos_consulta
        )

        filas_ranking.append({
            "Posición": posicion,
            "Identificador": identificador,
            "Título": titulos_documentos.get(
                identificador,
                "Título no encontrado"
            ),
            "Puntuación": puntuacion,
            "Fragmento": fragmento
        })

    return pd.DataFrame(filas_ranking)
rankings = {}

for indice_consulta, id_consulta in enumerate(ids_consultas):

    ranking = construir_ranking(
        indice_consulta=indice_consulta,
        similitudes=similitudes_manuales,
        ids_documentos=ids_documentos,
        documentos=documentos,
        titulos_documentos=titulos_documentos,
        consultas_procesadas=consultas_procesadas,
        top_k=5
    )

    # Guardamos la tabla por si queremos utilizarla después
    rankings[id_consulta] = ranking

    print(f"Consulta {id_consulta}: {consultas[id_consulta]['texto']}")
    print("Necesidad:", consultas[id_consulta]["necesidad"])

    display(ranking)

Consulta Q1: love and marriage
Necesidad: Encontrar textos sobre el amor y matrimonio


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.096663,"...r, yet ever full of faith, Consort to every..."
1,2,blake-poems.txt,Poems by William Blake 1789,0.094929,"...the noonday. ""And we are put on earth a lit..."
2,3,milton-paradise.txt,Paradise Lost by John Milton 1667,0.042289,"...y choose, Dilated or condensed, bright or o..."
3,4,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,0.023595,...e could receive; and the cheerfulness of th...
4,5,austen-persuasion.txt,Persuasion by Jane Austen 1818,0.022337,"...l, of steady age and character, and extreme..."


Consulta Q2: sea and whale
Necesidad: Encontrar textos relacionados con el mar y las ballenas


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick by Herman Melville 1851,0.470473,"...im of his mortality. ""While you take in han..."
1,2,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.048863,"...come, chanting the chant of battles, I abov..."
2,3,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryan...,0.030560,...some wings and a tail. THE CLOUD One hot su...
3,4,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton...,0.020508,"...f Dr Orion Hood, the eminent criminologist ..."
4,5,milton-paradise.txt,Paradise Lost by John Milton 1667,0.015645,"...rld, and all our woe, With loss of Eden, ti..."


Consulta Q3: king and kingdom
Necesidad: Encontrar textos sobre reyes y reinos


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryan...,0.077454,"...rown, dry, and desolate, from drought. The ..."
1,2,shakespeare-hamlet.txt,The Tragedie of Hamlet by William Shakespeare ...,0.075817,.... Barnardo. Who's there? Fran. Nay answer m...
2,3,bible-kjv.txt,The King James Bible,0.065730,...[The King James Bible] The Old Testament of...
3,4,shakespeare-macbeth.txt,The Tragedie of Macbeth by William Shakespeare...,0.034837,"...is faire, Houer through the fogge and filth..."
4,5,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carr...,0.029354,"...well as she could, for the hot day made her..."


Consulta Q4: children and family
Necesidad: Encontrar textos relacionados con niños y familia


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,austen-persuasion.txt,Persuasion by Jane Austen 1818,0.038711,...e printer's hands; but Sir Walter had impro...
1,2,bible-kjv.txt,The King James Bible,0.037296,"...an he said, I will greatly multiply thy sor..."
2,3,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton...,0.033577,"...ers called Todhunter. Well, Mr Brown, I am ..."
3,4,blake-poems.txt,Poems by William Blake 1789,0.029640,...is dwelling too. HOLY THURSDAY 'Twas on a H...
4,5,edgeworth-parents.txt,"The Parent's Assistant, by Maria Edgeworth",0.026792,"...e castle of Rossmore, in Ireland, is a smal..."


### Declaración de uso de IA
Durante la realización de este trabajo se han utilizado herramientas de inteligencia artificial con el objetivo de comprender con mayor profundidad la materia y conceptos necesarios para realizar la actividad, mejorar el código redactado, explicar y realizar pseudocódigo, investigar acerca de librerías y redacción en Python.